<a href="https://colab.research.google.com/github/Youseef3550/flyrank-ml-yousef-assighn-1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Youseef3550/flyrank-ml-yousef-assighn-1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — connect DuckDB to the warehouse

*Run this once. `HF_TOKEN` comes from a Colab Secret, never pasted into a cell (this repo is public).*

In [2]:
%pip -q install duckdb huggingface_hub

import duckdb, os

# Colab: Secrets panel (key icon) -> add HF_TOKEN. Falls back to getpass locally.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    import getpass
    HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Mid-panel iteration month -- NOT the sealed final month (2026-06), which is test-only.
MONTH = '2026-03'
print('Connected. Iterating on month =', MONTH)

Connected. Iterating on month = 2026-03


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**My lane: Refresh / Content Opportunity Scoring (Lane 2).**

**Unit of analysis (one row):** one row of `fact_content_daily_performance` is **one content item, for one client, on one calendar report day** — the grain is `report_date × client_hash_id × content_hash_id`. This is a daily performance record, not a page-level summary; a single content item contributes many rows (one per day it has activity in the panel).

**Time window:** I'm iterating on a single **mid-panel month, `month=2026-03`**, using the table's Hive partitioning (`fact_content_daily_performance/month=2026-03/*.parquet`) so DuckDB only reads that partition instead of scanning all ~79M rows. I deliberately avoid the `_sample` table for anything label-related — the warehouse README is explicit that `_sample` *is* the final month (June 2026), the sealed test window, and using it to develop label logic would mean training on the very outcome window I'll eventually be graded on.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Table(s) I'll use:** `fact_content_daily_performance` (primary — daily GSC/GA4 metrics), joined to `dim_content` (content-level context) and `dim_clients` (per-client panel start dates, so I don't treat "not tracked yet" as "zero traffic").

**What I'd predict or rank (label / proxy):** for the *capstone* version of this lane the target is a **forward-looking proxy**: did a content item's impressions in the 30 days *after* the decision point drop by 20%+ versus the 30 days *before* it (same idea as `w02`'s `is_declining_label`, but time-shifted so the label is a real future outcome instead of a same-window bucket). This notebook only builds the March feature frame; the label's forward window is used later, only in Part 3's leakage trap, to demonstrate the danger — it is **not** trained on for real here.

**Field buckets:**

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `gsc_avg_position`, `gsc_impressions`, `gsc_clicks`, derived `ctr`, `ga4_sessions` (gated on `ga4_data_available`) — all aggregated **within March only** | Every one of these is knowable by the end of the March decision window; none reach into April |
| **Label / proxy** | forward impressions change, April vs. March | The thing being predicted — never allowed into the feature set |
| **Context** | `client_hash_id`, `content_hash_id`, `report_date` | Join / group / filter keys only — pseudonymous ids carry no signal of their own |
| **Excluded** | `fact_content_query_90d`'s `impressions_90d` / `*_last30` columns | Its fixed 90-day window overlaps the most recent months of the snapshot, so for a label defined on a recent month these columns can contain the label period itself — leakage. I exclude them entirely from this lane rather than trying to only use `*_prev30`, to keep the March feature frame unambiguous |

## 3. Verify it with queries (grain, counts, missing values, windows) — then five features, then the trap

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3a. Query 1 — grain: is one row really `report_date × client × content`?

In [3]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f'Duplicate (report_date, client, content) combos in {MONTH}: {len(grain_check)}')
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (report_date, client, content) combos in 2026-03: 0


,report_date,client_hash_id,content_hash_id,n


*If the cell above prints 0 duplicate combos, the stated grain holds: one row really is one content item, for one client, on one day.*

### 3b. Query 2 — my slice's row count and date span

In [4]:
span = con.sql(f"""
    SELECT COUNT(*)                    AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_content_items,
           COUNT(DISTINCT client_hash_id)  AS n_clients,
           MIN(report_date)            AS min_date,
           MAX(report_date)            AS max_date
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
""").df()

span

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_content_items,n_clients,min_date,max_date
0,9841378,331437,55,2026-03-01,2026-03-31


*Expect the date span to sit fully inside March 2026, and the row count to be far smaller than the full 78.8M-row table — that's the whole point of filtering to one partition.*

### 3c. Query 3 — availability: filter with `IS TRUE`, how many rows survive?

In [5]:
availability = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4,
           ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)
                 / COUNT(*), 1) AS pct_with_ga4
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
""").df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_with_ga4,pct_with_ga4
0,9841378,413966.0,4.2


*`ga4_data_available IS TRUE` is the honest filter — rows before a client's GA4 tracking started have GA4 columns zero-**filled**, not genuinely zero. `IS TRUE` (rather than `= 1` or a truthy check) also survives any `NULL`s in the flag without silently dropping or keeping them by accident.*

### 3d. Five features (max) — build the March feature frame

Every feature below is aggregated **only from March rows** (`month = '2026-03'`), so nothing here can see April, the month I'll eventually rank against.

In [6]:
features = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        AVG(f.gsc_avg_position)                                   AS avg_position_mar,
        SUM(f.gsc_impressions)                                    AS total_impressions_mar,
        SUM(f.gsc_clicks)                                         AS total_clicks_mar,
        SUM(f.gsc_clicks) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS ctr_mar,
        SUM(CASE WHEN f.ga4_data_available IS TRUE
                 THEN f.ga4_sessions ELSE NULL END)                AS total_sessions_mar
    FROM {TABLES['fact_daily']} f
    WHERE f.month = '{MONTH}'
    GROUP BY 1, 2
    HAVING total_impressions_mar >= 10
""").df()

print(f'{len(features):,} content items with a March feature row')
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

143,206 content items with a March feature row


,client_hash_id,content_hash_id,avg_position_mar,total_impressions_mar,total_clicks_mar,ctr_mar,total_sessions_mar
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,7.209549,6523.0,7.0,0.001073,1.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2.987198,453.0,0.0,0.000000,NaN
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,6.724039,5630.0,6.0,0.001066,3.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,7.244844,4944.0,13.0,0.002629,2.0
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,14.432540,42.0,0.0,0.000000,7.0


**Feature list — "knowable at the decision moment because…"**

1. **`avg_position_mar`** — average GSC ranking position across March. Knowable because every underlying `report_date` is on or before the end-of-March decision point.
2. **`total_impressions_mar`** — summed GSC impressions across March. Knowable for the same reason: it's a closed, past window by the time I'd act on it.
3. **`total_clicks_mar`** — summed GSC clicks across March. Same closed-window logic as impressions.
4. **`ctr_mar`** — clicks ÷ impressions, computed only from the two March aggregates above. Knowable because it's arithmetic on numbers that already exist by decision time — no future data enters the ratio.
5. **`total_sessions_mar`** — GA4 sessions in March, gated on `ga4_data_available IS TRUE` so clients without GA4 tracking yet contribute `NULL`, not a false zero. Knowable because it's logged engagement data from the same closed March window.

### 3e. The trap — one label-derived column, on purpose

Now I deliberately break the rule I just followed: I pull in **April** impressions, define a forward-decline label from March→April, and then "accidentally" leave the raw April number sitting in the feature set. Watch the score.

In [7]:
# Pull April (the month right after the feature window) — only to build the label.
april = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS total_impressions_apr
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-04'
    GROUP BY 1, 2
""").df()

trap = features.merge(april, on=['client_hash_id', 'content_hash_id'], how='inner')
trap = trap.dropna(subset=['avg_position_mar', 'ctr_mar'])

# The label: did impressions drop 20%+ from March to April? (a forward outcome, as it should be)
trap['declined'] = (trap['total_impressions_apr'] < 0.8 * trap['total_impressions_mar']).astype(int)
print('Base rate (share declined):', trap['declined'].mean().round(3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Base rate (share declined): 0.517


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ['avg_position_mar', 'total_impressions_mar', 'total_clicks_mar', 'ctr_mar']
leaky_features  = honest_features + ['total_impressions_apr']   # <- the label-derived column

def quick_score(feature_cols, label='trap run'):
    X = trap[feature_cols].fillna(0)
    y = trap['declined']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    print(f'{label:22} AUC = {auc:.3f}   (features: {feature_cols})')
    return auc

honest_auc = quick_score(honest_features, 'honest (March only)')
leaky_auc  = quick_score(leaky_features,  'WITH april leak')

honest (March only)    AUC = 0.595   (features: ['avg_position_mar', 'total_impressions_mar', 'total_clicks_mar', 'ctr_mar'])
WITH april leak        AUC = 1.000   (features: ['avg_position_mar', 'total_impressions_mar', 'total_clicks_mar', 'ctr_mar', 'total_impressions_apr'])


**What happened:** adding `total_impressions_apr` — a column computed from the exact same April window the label is derived from — pushed AUC from 0.595 to a perfect 1.000, because the model isn't finding a pattern, it's reading the answer off a sibling of the label itself (`declined = impressions_apr < 0.8 * impressions_mar`). That collapse-on-removal is exactly the leakage confession the `hunting-leakage-and-validating` skill describes: train once WITH the suspect, once WITHOUT, and a drop from near-perfect back down to a modest, believable number is the tell.

**Deleting the trap and keeping the honest number:** the real result of this lane, going forward, is the **`honest_auc`** printed above — March-only features, nothing from the label window. `leaky_features` and `total_impressions_apr` are never used again after this cell.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation: the panel is unbalanced across clients, and March is not the same March for everyone.** `dim_clients.gsc_data_start` shows tracking histories that differ wildly between clients — some have well over a year of data, others only a few months. A handful of clients in my March slice may have only just started being tracked that month, which means their `total_impressions_mar` and `total_sessions_mar` reflect a partial ramp-up, not a representative month of normal performance. I can't currently tell "a client having a genuinely quiet March" apart from "a client whose tracking only turned on mid-March" without joining back to `dim_clients` and explicitly checking `gsc_data_start`/`ga4_data_start` against the window — which this notebook doesn't yet do. Any ranking built on this March frame should be read as decision-support only, and re-checked per client before trusting a single month's numbers for a client near the edge of their tracking start.

In [9]:
recent_start = con.sql(f"""
    SELECT COUNT(*) AS clients_started_in_march
    FROM {TABLES['dim_clients']}
    WHERE gsc_data_start BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
""").df()

recent_start

,clients_started_in_march
0,5


## 5. Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.